# Tahap 2 — Case Representation

Mengekstrak metadata dan fitur dari setiap putusan yang sudah dibersihkan, lalu menyimpannya dalam format terstruktur.

**Input**: `data/processed/clean/pasal_362/*.txt` dan `data/processed/clean/pasal_363/*.txt`

**Output**: `data/processed/cases.csv` dan `data/processed/cases.json`

**Kolom**: `case_id`, `filename`, `label_pasal`, `no_perkara`, `tanggal`, `terdakwa`, `pasal_dakwaan`, `amar_putusan`, `vonis_bulan` (integer bulan), `ringkasan_fakta`, `argumen_hukum`, `word_count`, `text_full`

## 2.1 Import Library

In [1]:
import re
import json
import logging
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

## 2.2 Konfigurasi

In [2]:
CLEAN_DIR  = Path("../data/processed/clean")
OUTPUT_DIR = Path("../data/processed")
LOG_DIR    = Path("../logs")

for d in [OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "representation.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

## 2.3 Fungsi Ekstraksi Metadata

In [3]:
def ekstrak_no_perkara(text):
    patterns = [
        r"\d+\s*/\s*pid\.b\s*/\s*\d{4}\s*/\s*pn\s*[\w\.]+",
        r"\d+\s*/\s*pid\.sus\s*/\s*\d{4}\s*/\s*pn\s*[\w\.]+",
        r"nomor\s*[:\s]+(\d+[/\-][\w\.]+[/\-]\d{4}[/\-][\w\.]+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return ""


def ekstrak_tanggal(text):
    bulan_map = {
        "januari": "01", "februari": "02", "maret": "03", "april": "04",
        "mei": "05", "juni": "06", "juli": "07", "agustus": "08",
        "september": "09", "oktober": "10", "november": "11", "desember": "12"
    }
    pat = r"(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+(\d{4})"
    m = re.search(pat, text, re.IGNORECASE)
    if m:
        hari, bln, thn = m.group(1), m.group(2).lower(), m.group(3)
        return f"{thn}-{bulan_map.get(bln, '00')}-{int(hari):02d}"
    return ""


def ekstrak_terdakwa(text):
    patterns = [
        r"terdakwa\s*[:\s]+([A-Z][a-zA-Z\s]+?)(?:\s*,|\s*alias|\s*bin|\s*binti|\n)",
        r"nama\s*lengkap\s*[:\s]+([A-Z][a-zA-Z\s]+?)(?:\s*;|\s*,|\n)",
        r"nama\s*[:\s]+([A-Z][a-zA-Z\s\.]+?)(?:\s*;|\s*,|\s*\n)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            nama = m.group(1).strip()
            if 3 < len(nama) < 60:
                return nama.title()
    return ""


def ekstrak_pasal_dakwaan(text):
    patterns = [
        r"pasal\s+362\s+kuhp",
        r"pasal\s+363\s+(?:ayat\s+\(\d\)\s+)?kuhp",
        r"pasal\s+364\s+kuhp",
        r"pasal\s+365\s+kuhp",
        r"pasal\s+3[56789]\d?\s+(?:jo\.?\s+pasal\s+\d+\s+)?kuhp",
    ]
    found = []
    for pat in patterns:
        matches = re.findall(pat, text, re.IGNORECASE)
        found.extend(matches)
    return "; ".join(set(found)) if found else ""


def ekstrak_amar_putusan(text):
    m = re.search(
        r"m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i(.{50,800}?)(?=menimbang|menetapkan|demikian|$)",
        text, re.IGNORECASE | re.DOTALL
    )
    if m:
        blok = m.group(1).lower().strip()
        if re.search(r"terbukti.*bersalah|menyatakan.*terdakwa.*bersalah", blok):
            return "bersalah"
        if re.search(r"tidak terbukti|membebaskan terdakwa", blok):
            return "bebas"
        if re.search(r"melepaskan terdakwa|lepas dari", blok):
            return "lepas"
        return blok[:200].replace("\n", " ").strip()
    return ""


def ekstrak_vonis_bulan(text):
    """Ekstrak vonis dalam satuan bulan (integer). Mengembalikan 0 jika tidak ditemukan."""
    tahun_m = re.search(r"pidana penjara selama\s+(\d+)\s*\([\w\s]+\)\s*tahun", text, re.IGNORECASE)
    bulan_m = re.search(r"pidana penjara selama\s+(\d+)\s*\([\w\s]+\)\s*bulan", text, re.IGNORECASE)
    if not tahun_m and not bulan_m:
        tahun_m = re.search(r"penjara\s+(\d+)\s+tahun", text, re.IGNORECASE)
        bulan_m = re.search(r"penjara\s+(\d+)\s+bulan", text, re.IGNORECASE)
    total = 0
    if tahun_m:
        total += int(tahun_m.group(1)) * 12
    if bulan_m:
        total += int(bulan_m.group(1))
    return total


def ekstrak_ringkasan_fakta(text):
    patterns = [
        r"menimbang.*?bahwa(.{100,600}?)(?=menimbang|mempertimbangkan|mengenai|$)",
        r"bahwa terdakwa(.{100,500}?)(?=bahwa|menimbang|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            snippet = re.sub(r"\s+", " ", m.group(1).replace("\n", " ").strip())
            return snippet[:500]
    return ""


def ekstrak_argumen_hukum(text):
    patterns = [
        r"mempertimbangkan(.{100,600}?)(?=menimbang|mengadili|menetapkan|$)",
        r"pertimbangan hukum(.{100,500}?)(?=mengadili|menetapkan|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            snippet = re.sub(r"\s+", " ", m.group(1).replace("\n", " ").strip())
            return snippet[:500]
    return ""

## 2.4 Proses Ekstraksi

In [4]:
txt_files = sorted(CLEAN_DIR.rglob("*.txt"))
print(f"Total file clean ditemukan: {len(txt_files)}")

rows = []

for txt_path in tqdm(txt_files, desc="Ekstraksi metadata"):
    label_folder = txt_path.parent.name
    text = txt_path.read_text(encoding="utf-8", errors="ignore")

    row = {
        "case_id":         txt_path.stem,
        "filename":        txt_path.name,
        "label_pasal":     label_folder,
        "no_perkara":      ekstrak_no_perkara(text),
        "tanggal":         ekstrak_tanggal(text),
        "pengadilan":      "PN Malang",
        "terdakwa":        ekstrak_terdakwa(text),
        "pasal_dakwaan":   ekstrak_pasal_dakwaan(text),
        "amar_putusan":    ekstrak_amar_putusan(text),
        "vonis_bulan":     ekstrak_vonis_bulan(text),
        "ringkasan_fakta": ekstrak_ringkasan_fakta(text),
        "argumen_hukum":   ekstrak_argumen_hukum(text),
        "word_count":      len(text.split()),
        "text_full":       text[:3000],
    }
    rows.append(row)

print(f"Total kasus diekstrak: {len(rows)}")

Total file clean ditemukan: 60


Ekstraksi metadata:   0%|          | 0/60 [00:00<?, ?it/s]

Total kasus diekstrak: 60


## 2.5 Simpan ke CSV dan JSON

In [5]:
df = pd.DataFrame(rows)

csv_path = OUTPUT_DIR / "cases.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"CSV disimpan: {csv_path} ({len(df)} baris)")

json_path = OUTPUT_DIR / "cases.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f"JSON disimpan: {json_path}")

CSV disimpan: ..\data\processed\cases.csv (60 baris)
JSON disimpan: ..\data\processed\cases.json


## 2.6 Distribusi Label dan Preview

In [6]:
print("Distribusi label_pasal:")
print(df["label_pasal"].value_counts().to_string())

print(f"\nDistribusi vonis_bulan (statistik):")
print(df["vonis_bulan"].describe().to_string())

vonis_zero = (df["vonis_bulan"] == 0).sum()
print(f"\nKasus dengan vonis tidak terdeteksi (0 bulan): {vonis_zero}")
print(f"Kasus dengan vonis terdeteksi: {len(df) - vonis_zero}")

print(f"\nRata-rata word count: {df['word_count'].mean():.0f} kata")

preview_cols = ["case_id", "label_pasal", "no_perkara", "tanggal", "amar_putusan", "vonis_bulan", "word_count"]
df[preview_cols].head(10)

Distribusi label_pasal:
label_pasal
pasal_362    30
pasal_363    30

Distribusi vonis_bulan (statistik):
count    60.000000
mean     15.333333
std      16.547483
min       0.000000
25%       0.000000
50%      12.000000
75%      24.000000
max      72.000000

Kasus dengan vonis tidak terdeteksi (0 bulan): 18
Kasus dengan vonis terdeteksi: 42

Rata-rata word count: 7056 kata


,case_id,label_pasal,no_perkara,tanggal,amar_putusan,vonis_bulan,word_count
0,case_pasal_362_001,pasal_362,132/pid.b/2026/pn mlgdemi,1986-09-21,bersalah,12,4500
1,case_pasal_362_002,pasal_362,108/pid.b/2026/pn mlg,1986-09-21,bersalah,12,6278
2,case_pasal_362_003,pasal_362,119/pid.b/2026/pn mlgdemi,1997-06-29,bersalah,24,8571
3,case_pasal_362_004,pasal_362,96/pid.b/2026/pn mlgdemi,2001-02-03,bersalah,12,3353
4,case_pasal_362_005,pasal_362,17/pid.b/2026/pn mlgdemi,1985-09-10,bersalah,24,8909
5,case_pasal_362_006,pasal_362,433/pid.b/2025/pn mlgdemi,1999-08-26,,72,20589
6,case_pasal_362_007,pasal_362,58/pid.b/2026/pn mlgdemi,1972-01-01,bersalah,5,7324
7,case_pasal_362_008,pasal_362,55/pid.b/2026/pn mlgdemi,1986-04-26,bersalah,10,4576
8,case_pasal_362_009,pasal_362,87/pid.b/2026/pn mlgdemi,1985-11-10,,12,4762
9,case_pasal_362_010,pasal_362,45/pid.b/2026/pn mlgdemi,2005-05-26,bersalah,12,5562
